# RNN Text Classification of Customer Messages
Bidirectional LSTM that classifies short customer messages into 77 categories.

Run this notebook from the `notebooks/` folder.

## 1. Load and explore the data

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

train = pd.read_csv('../dataset/customer_train.csv')
test = pd.read_csv('../dataset/customer_test.csv')
print(train.shape, test.shape)
train.sample(5, random_state=1)

In [ ]:
train['n_words'] = train['text'].str.split().str.len()
print(train['n_words'].describe())
train['n_words'].hist(bins=40); plt.xlabel('words per message'); plt.show()

train['category'].value_counts().head(10)

## 2. Build the RNN model
Text -> `TextVectorization` -> `Embedding` -> `Bidirectional(LSTM)` -> `Dense(softmax)`.

In [ ]:
from tensorflow.keras import layers, models, callbacks

MAX_TOKENS, SEQ_LEN = 8000, 40
le = LabelEncoder().fit(train['category'])
y_train, y_test = le.transform(train['category']), le.transform(test['category'])
X_train = tf.constant(train['text'].astype(str).to_numpy())
X_test = tf.constant(test['text'].astype(str).to_numpy())

vec = layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN)
vec.adapt(X_train)

inp = layers.Input(shape=(1,), dtype=tf.string)
x = vec(inp)
x = layers.Embedding(MAX_TOKENS, 128, mask_zero=True)(x)
x = layers.Bidirectional(layers.LSTM(128))(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(le.classes_), activation='softmax')(x)
model = models.Model(inp, out)
model.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 3. Train

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.1, epochs=25, batch_size=64,
                    callbacks=[callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)],
                    verbose=2)
pd.DataFrame(history.history)[['accuracy','val_accuracy']].plot(); plt.show()

## 4. Evaluate on the unseen test set

In [ ]:
pred = model.predict(X_test, verbose=0).argmax(axis=1)
print('Test accuracy:', accuracy_score(y_test, pred))

## 5. Predict previously unseen messages

In [ ]:
new_msgs = [
    'My new card still has not shown up in the mail',
    'How do I change my PIN number?',
    'I was charged twice for the same purchase',
    'Why was my transfer declined?',
]
probs = model.predict(tf.constant(new_msgs), verbose=0)
for m, p in zip(new_msgs, probs):
    top = p.argsort()[::-1][:3]
    print(m)
    for i in top: print(f'   {le.classes_[i]:<35s} {p[i]*100:5.1f}%')

## 6. Where does the model go wrong?

In [ ]:
test['pred'] = le.classes_[pred]
wrong = test[test['pred'] != test['category']]
print(f'{len(wrong)} wrong of {len(test)}')
wrong.sample(10, random_state=1)[['text','category','pred']]